In [ ]:
"""
FixMaster Inference Test Script
---------------------------------
Loads the fine-tuned LoRA adapters on top of the quantized base model
and runs a batch of test bug/error examples through it, printing the
extracted git diff for each.

"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B"

# Path to your final saved adapter folder (adjust if different)
ADAPTER_DIR = "/teamspace/studios/this_studio/fixmaster-lora-v3"

# -----------------------------------------------------------------------
# Test cases across a few languages / bug types to check generalization
# -----------------------------------------------------------------------
TEST_CASES = [
    {
        "language": "Java",
        "error_log": "NullPointerException: Attempt to invoke virtual method on a null object reference",
        "buggy_code": "public void displayUser(User user) {\n    System.out.println(user.getName());\n}",
    },
    {
        "language": "Python",
        "error_log": "Bandit: B608: Hardcoded SQL string detected.",
        "buggy_code": "def get_user(db, user_id):\n    query = f'SELECT * FROM users WHERE id = {user_id}'\n    return db.execute(query)",
    },
    {
        "language": "JavaScript",
        "error_log": "TypeError: Cannot read properties of undefined (reading 'map')",
        "buggy_code": "function renderList(items) {\n    return items.map(i => `<li>${i}</li>`);\n}",
    },
    {
        "language": "C",
        "error_log": "Segmentation fault (core dumped)",
        "buggy_code": "int arr[5];\nfor(int i=0; i<=5; i++) {\n    arr[i] = 0;\n}",
    },
    {
        "language": "Go",
        "error_log": "panic: runtime error: index out of range [5] with length 5",
        "buggy_code": "func getItem(items []string, idx int) string {\n    return items[idx]\n}",
    },
]

SYSTEM_PROMPT = (
    "You are FixMaster, an autonomous security remediation agent. "
    "Given an error log and buggy code, output ONLY the unified Git patch. "
    "Do not output conversational text."
)


def build_prompt(language: str, error_log: str, buggy_code: str) -> str:
    return f"""<|im_start|>system
{SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
Language: {language}
Error: {error_log}
Code:
{buggy_code}<|im_end|>
<|im_start|>assistant
```diff
"""


def extract_patch(generated_text: str) -> str:
    assistant_prefix = "<|im_start|>assistant\n```diff\n"
    if assistant_prefix not in generated_text:
        return "Could not extract patch. Full generated text:\n" + generated_text

    patch_start = generated_text.find(assistant_prefix) + len(assistant_prefix)
    patch_text = generated_text[patch_start:].strip()

    if "<|im_end|>" in patch_text:
        patch_text = patch_text[: patch_text.find("<|im_end|>")].strip()
    if "```" in patch_text:
        patch_text = patch_text[: patch_text.find("```")].strip()

    return patch_text


def main():
    print(f"Loading tokenizer from {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token

    print("Configuring 4-bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    print(f"Loading base model {MODEL_ID}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    print(f"Loading LoRA adapters from {ADAPTER_DIR}...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model.eval()

    print("\n" + "=" * 70)
    print(f"Running {len(TEST_CASES)} test cases...")
    print("=" * 70)

    for i, case in enumerate(TEST_CASES, start=1):
        prompt = build_prompt(case["language"], case["error_log"], case["buggy_code"])
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_new_tokens=300,
                do_sample=False,
                repetition_penalty=1.3,       # penalizes repeated tokens
                no_repeat_ngram_size=4,       # hard-blocks any 4-gram from repeating
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
        patch = extract_patch(generated_text)

        print(f"\n--- Test {i}: {case['language']} ---")
        print(f"Error: {case['error_log']}")
        print(f"Buggy code:\n{case['buggy_code']}")
        print("\nGenerated patch:")
        print(patch)
        print("-" * 70)

    print("\nAll test cases complete.")


if __name__ == "__main__":
    main()